In [13]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import copy

import scipy.stats as stats
import re

import os
%config InlineBackend.figure_format ='retina'

%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../../")
import builtins
    
builtins.NAMING = "new"

from RealTrace_paper_analysis.plotting_src.read_ggp_run import * # you might need to change these two line, such that you can use read ggp_run
from RealTrace_paper_analysis.simulations.cells_simulation import * # you might need to change these two line, such that you can use read ggp_run

from RealTrace_paper_analysis.plotting_src.cells_utils import *
from RealTrace_paper_analysis.plotting_src.read_utils import *
from RealTrace_paper_analysis.plotting_src.header_settings import *
from RealTrace_paper_analysis.plotting_src.plot_correlations_from_npz import *

from RealTrace_paper_analysis.plotting_src.naive_rates import *

from mpl_toolkits.axes_grid1.inset_locator import (inset_axes, InsetPosition,
                                                  mark_inset)
import matplotlib
%config InlineBackend.figure_format ='retina'


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [56]:
filenames = ["../../ecoli_other_labs/Kar2021/alanine_upload.txt",
             "../../ecoli_other_labs/Kar2021/glycerol_upload.txt",
             "../../ecoli_other_labs/Kar2021/glucose_cas_upload.txt"]

# for file in filenames:
#     process_file(file)

filenames_extended = [f.replace("_upload.txt", "_extended_upload.txt") for f in filenames]

In [140]:
def find_parent(meas, extended):
    meas_str = "_".join( np.round(np.diff(meas)[:10], 2).astype(str))
    for cell in extended.keys():
        if meas_str in extended[cell]:
            return cell
    


def get_time_diff(data_df):
    unique_Tns = np.unique(data_df[data_df["t-BirthFrame"] == "Tn"].to_numpy()[0][2:])
    return np.min(np.diff(np.sort(unique_Tns)))



def process_cell(raw_data_row, dt):
    data = raw_data_row[2:]
    meas = data[np.isfinite(data)].to_numpy()
    time = np.arange(len(meas))*dt + raw_data_row[1]
    return time, meas


def process_extended_cell(raw_data_row):
    data = raw_data_row[3:]
    meas = data[np.isfinite(data)].to_numpy()
    return meas


def process_file(filename):
    outfile = filename.replace("_upload.txt", "_processed.csv")
    raw_data = pd.read_csv( filename ,delimiter="\t")
    
    cell_ids = raw_data.columns[1:]
    dt = get_time_diff(raw_data)

    # extended_data = pd.read_csv( filename.replace("_upload.txt", "_extended_upload.txt"),
    #                         delimiter="\t")
    # extended_data_dict ={}
    # for cell_id in cell_ids:
    #     if cell_id in extended_data.keys():
    #         meas = process_extended_cell(extended_data[cell_id])
    #         print(np.round(np.diff(meas), 2))
    #         extended_data_dict[cell_id] = "_".join( np.round(np.diff(meas), 2).astype(str))
    #     else:
    #         pass    
            
    dataset = pd.DataFrame()
    
    for cell_id in cell_ids[:100]:
        time, meas = process_cell(raw_data[cell_id], dt)
        
        cell_df = pd.DataFrame({ "time":time, "length_um": meas})
        cell_df["cell_id"]=cell_id
        
        dataset = pd.concat([dataset, cell_df], ignore_index=True)
        
    return
    dataset["dummy_gfp"] = np.random.normal(loc=0, scale=1e-3, size=len(dataset["time"]))
    dataset["parent_id"] = -1
    dataset.to_csv(outfile)

for file in filenames:
    _= process_file(file)
    break

In [3]:
for file in filenames: 
    print(file, "\n")
    out_dir = os.path.join(*file.split("/")[:-1], "RealTrace_out")
    outfile = file.replace("_upload.txt", "_processed.csv")
    
    param_file = os.path.join(*file.split("/")[:-1], "parameters.txt")
    config_file = os.path.join(*file.split("/")[:-1], "csv_config.txt")
    
    run_cmd = "../../RealTrace/bin/RealTrace" + \
            " -c " + config_file + \
            " -b " + param_file + \
            " -t " + str(1e-40) + \
            " -i " + outfile + \
            " -o " + out_dir + \
            " -noise const" + \
            " -space log" +\
            " -m -p -j"
    print("run: -> \n", run_cmd)
    # s = subprocess.run(run_cmd, capture_output=True, shell=True)
    # print(s)

../../ecoli_other_labs/Kar2021/alanine_upload.txt 

run: -> 
 ../../RealTrace/bin/RealTrace -c ../../ecoli_other_labs/Kar2021/csv_config.txt -b ../../ecoli_other_labs/Kar2021/parameters.txt -t 1e-40 -i ../../ecoli_other_labs/Kar2021/alanine_processed.csv -o ../../ecoli_other_labs/Kar2021/RealTrace_out -noise const -space log -m -p -j
../../ecoli_other_labs/Kar2021/glycerol_upload.txt 

run: -> 
 ../../RealTrace/bin/RealTrace -c ../../ecoli_other_labs/Kar2021/csv_config.txt -b ../../ecoli_other_labs/Kar2021/parameters.txt -t 1e-40 -i ../../ecoli_other_labs/Kar2021/glycerol_processed.csv -o ../../ecoli_other_labs/Kar2021/RealTrace_out -noise const -space log -m -p -j
../../ecoli_other_labs/Kar2021/glucose_cas_upload.txt 

run: -> 
 ../../RealTrace/bin/RealTrace -c ../../ecoli_other_labs/Kar2021/csv_config.txt -b ../../ecoli_other_labs/Kar2021/parameters.txt -t 1e-40 -i ../../ecoli_other_labs/Kar2021/glucose_cas_processed.csv -o ../../ecoli_other_labs/Kar2021/RealTrace_out -noise const -s